In [15]:
import os
import json
import re
import pdfplumber
import pytesseract
import cv2
import numpy as np
from PIL import Image
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

pytesseract.pytesseract.tesseract_cmd = r"C:\Users\pkeer\Downloads\tesseract-ocr-w64-setup-5.5.0.20241111.exe"


In [3]:
def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text


In [4]:
def extract_text_from_image(img_path):
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    text = pytesseract.image_to_string(gray)
    return text


In [5]:
def get_text(file_path):
    if file_path.endswith(".pdf"):
        return extract_text_from_pdf(file_path)
    elif file_path.endswith((".png", ".jpg", ".jpeg")):
        return extract_text_from_image(file_path)
    elif file_path.endswith(".txt"):
        return open(file_path, "r", encoding="utf-8").read()
    else:
        return ""


In [16]:
def extract_medical_json(ocr_text):

    prompt = f"""
You are a medical document information extraction system.

Extract structured medical information from the OCR text.

Return ONLY valid JSON in this schema:

{{
 "patient_name": "",
 "age": "",
 "gender": "",
 "test_date": "",
 "hemoglobin": "",
 "blood_sugar_fasting": "",
 "blood_sugar_postprandial": "",
 "cholesterol_total": "",
 "hdl": "",
 "ldl": "",
 "triglycerides": "",
 "bmi": "",
 "blood_pressure": "",
 "diagnosis": [],
 "doctor_notes": "",
 "abnormal_flags": []
}}

OCR TEXT:
{ocr_text}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content


In [17]:
file_path = r"C:\Users\pkeer\OneDrive\Desktop\Infosys\sample.pdf"

print("Reading file...")

ocr_text = get_text(file_path)

print("OCR TEXT LENGTH:", len(ocr_text))

print("\n--- OCR PREVIEW ---\n")
print(ocr_text[:500])   # show first 500 chars

print("\nCalling LLM...")

json_output = extract_medical_json(ocr_text)

print("\n--- JSON OUTPUT ---\n")
print(json_output)


Reading file...
OCR TEXT LENGTH: 20124

--- OCR PREVIEW ---

.
Name : DUMMY
Lab No. : 439854467 Age : 30 Years
Ref By : U Gender : Male
Collected : 14/5/2023 11:03:00AM Reported : 16/5/2023 1:36:25PM
A/c Status : P Report Status : Final
Collected at : L P L-ROHINI (NATIONAL REFERENCE LAB) Processed at : L P L-NATIONAL REFERENCE LAB
National Reference laboratory, Block E, Sector National Reference laboratory, Block E,
18, ROHINI Sector 18, Rohini, New Delhi -110085
DELHI 110085
Test Report
Test Name Results Units Bio. Ref. Interval
SwasthFit Super 4
COMPLE

Calling LLM...

--- JSON OUTPUT ---

```json
{
  "patient_name": "DUMMY",
  "age": "30",
  "gender": "Male",
  "test_date": "14/5/2023",
  "hemoglobin": "15.00",
  "blood_sugar_fasting": "90.00",
  "blood_sugar_postprandial": "",
  "cholesterol_total": "105.00",
  "hdl": "46.00",
  "ldl": "33.00",
  "triglycerides": "130.00",
  "bmi": "",
  "blood_pressure": "",
  "diagnosis": [],
  "doctor_notes": "",
  "abnormal_flags": []
}
```


In [18]:
with open("output.json", "w") as f:
    f.write(json_output)

print("Saved → output.json")


Saved → output.json
